# Embedding demo/test

Example flow showing two methods of embedding signals into fixed-length intervals. First method (`FSDBData`) uses custom C++ program to embed signal activity directly from FSDB file, whereas second method (`NPI`) uses Verdi's built-in API method. While the former is significantly faster, the latter method is more intuitive by utilizing many useful Pandas abstractions/functions and is used to validate the first method.

There are three supported embeddings: `trans`, `toggles`, and `ones`. While `trans` and `toggles` shows a perfect match, `ones` has a slight mismatch between the two embedding methods for buses ONLY due to rounding errors.

### Setup

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()/".."))
import numpy as np
import pandas as pd

import fsdbparse
from fsdbparse.fsdbparse import FSDBParse, FSDBData
from fsdbparse.npi_wrapper import NPI


In [2]:
rtl = 'Boom3w800MHz'
fsdbfile = '/bwrcq/scratch/nayiri/power/chipyard-intech16-feb25/vlsi/output/chipyard.harness.TestHarness.Boom3w800MHz/towers.fsdb'
cache_dpath = Path.cwd()/"out"
embedding = 'toggles'
embedding = 'trans'
# embedding = 'ones'
cache_dpath

PosixPath('/bwrcq/C/nayiri/fsdb-parse/flows/out')

In [3]:
# from importlib import reload
# import fsdbparse
# reload(fsdbparse)
fsdbdata = FSDBData(fsdbfile,rtl=rtl,
                    cache_dpath=cache_dpath,
                    end_time=2500,
                    windowlen=1250,embedding=embedding,
                    print_only=False,
                    verbose=True)
# fsdb_emb = fsdbdata.get_embedding()
fsdbdata.siginfo.shape #, fsdb_emb.shape

(119245, 2)

## Signal Subset

In [4]:
# use NPI for more signal info
npi = NPI(fsdbfile)
all_sigs = npi.get_all_sigs(top_scope="TestDriver.testHarness.chiptop0")
len(all_sigs)

logDir = /bwrcq/C/nayiri/fsdb-parse/flows/npiLog

                     NPI - Native Programming Interface (R)

                Version W-2024.09-1 for linux64 - Oct 12, 2024 

                    Copyright (c) 1999 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity with the applicable License Key for such
  Licensed Products. Synopsys will use information gathered in connection with
    this process to deliver software updates and pursue software pirates and
                                   infringers.

 Inclus

139712

In [5]:
# generate siginfo for sig subset
sigs_subset = [s.full_name() for s in all_sigs if '_nack' in s.name()]
df_sigset = fsdbdata.generate_siginfo_sigset('all_nack',sigs_subset)
len(sigs_subset),len(df_sigset),df_sigset['sigwidth'].sum()

(60, 44, 67)

In [9]:
# generate embedding for sig subset
import importlib
importlib.reload(fsdbparse.fsdbparse)
from fsdbparse.fsdbparse import FSDBParse, FSDBData
fsdbdata_emb = FSDBData(fsdbfile,rtl=rtl,
                        cache_dpath=cache_dpath,
                        windowlen=1250,embedding=embedding,
                        sigset='all_nack',
                        print_only=False,
                        verbose=True)
dfemb_1 = fsdbdata_emb.get_embedding()
dfemb_1

towers (346, 44)...

signame,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_hellaCacheArb_io_requestor_0_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/ptw/io_mem_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_requestor_0_s2_nack","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_lsu_io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_mem_s2_nack",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid_0,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_valid","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_ldq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_stq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_ldq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_stq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_is_hella",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/s1_send_resp_or_nack_0,...,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_14,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_15,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_16,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_17,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_18,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_19,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_20,TestDriver/testHarness/chi

In [22]:
# generate embedding for sig subset
import importlib
importlib.reload(fsdbparse)
from fsdbparse.fsdbparse import FSDBParse, FSDBData
fsdbdata_emb = FSDBData(fsdbfile,rtl=rtl,
                        cache_dpath=cache_dpath,
                        windowlen=1250,embedding=embedding,
                        sigset='all_nack',
                        # print_only=True,
                        overwrite=True,
                        verbose=True)
dfemb_1 = fsdbdata_emb.get_embedding()
dfemb_1

# Dumping trans for towers
/bwrcq/C/nayiri/fsdb-parse/flows/../fsdbparse/dump_embedding --fsdb /bwrcq/scratch/nayiri/power/chipyard-intech16-feb25/vlsi/output/chipyard.harness.TestHarness.Boom3w800MHz/towers.fsdb --idcodes /bwrcq/C/nayiri/fsdb-parse/flows/out/siginfo/Boom3w800MHz.sigs-all_nack/idcodes.txt --windowlen 1250 --embedding trans --verbose --out /bwrcq/C/nayiri/fsdb-parse/flows/out/trans/Boom3w800MHz.sigs-all_nack/towers.windowlen-1250.bin


FSDB Reader, Release Verdi_W-2024.09-1, RH Linux x86_64/64bit, 11/03/2025
(C) 1996 - 2025 by Synopsys, Inc.
SCALE = 1, unit = ps
FSDB timescale: 1ps, _ ns x 1000 --> <timescale>
Embeddings: toggles-0, ones-0, trans-
Start time: 0 ps
End time:   433538000 ps
Time window length:   1250000 ps
Number of signals to process: 44
Number of time windows: 346
Array shape: (44,346)
Read design hierarchy.
Added signals to list.


fsdb_file = /bwrcq/scratch/nayiri/power/chipyard-intech16-feb25/vlsi/output/chipyard.harness.TestHarness.Boom3w800MHz/towers.fsdb
idcodes_file = /bwrcq/C/nayiri/fsdb-parse/flows/out/siginfo/Boom3w800MHz.sigs-all_nack/idcodes.txt
bin_file = /bwrcq/C/nayiri/fsdb-parse/flows/out/trans/Boom3w800MHz.sigs-all_nack/towers.windowlen-1250.bin
start_time = 0
end_time = 0
windowlen = 1250
embedding = trans
logDir = /tmp/
Writing compressed embedding w/ shape (346, 44) -> /bwrcq/C/nayiri/fsdb-parse/flows/out/trans/Boom3w800MHz.sigs-all_nack/towers.windowlen-1250.npz
towers (346, 44)...

Loaded signals.
Reading idcodes from: /bwrcq/C/nayiri/fsdb-parse/flows/out/siginfo/Boom3w800MHz.sigs-all_nack/idcodes.txt
Writing to file: /bwrcq/C/nayiri/fsdb-parse/flows/out/trans/Boom3w800MHz.sigs-all_nack/towers.windowlen-1250.bin


signame,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_hellaCacheArb_io_requestor_0_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/ptw/io_mem_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_requestor_0_s2_nack","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_lsu_io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_mem_s2_nack",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid_0,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_valid","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_ldq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_stq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_ldq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_stq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_is_hella",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/s1_send_resp_or_nack_0,...,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_14,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_15,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_16,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_17,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_18,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_19,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_20,TestDriver/testHarness/chi

In [24]:
import importlib
importlib.reload(fsdbparse.npi_wrapper)
from fsdbparse.npi_wrapper import NPI
# generate embedding with NPI (much slower)
npi = NPI(fsdbfile)
dfs = []
for sns in df_sigset['signame']:
    # if not sns.endswith('io_dmem_nack_0_bits_is_hella'): continue
    sn = sns.split(',')[0].split('[')[0].replace('/','.')
    df = npi.embed_signal(sn,start_ns=0,window_ns=1250,embedding=embedding,merge_buses=True)
    dfs.append(df)

dfemb_2 = pd.concat(dfs,axis=1).fillna(0).astype(np.uint16)
# grps = dfemb_2_bits.columns.str.split('-bit').str[0]
# dfemb_2 = dfemb_2_bits.groupby(grps,axis=1).sum().sort_index()
sn2fullsn = {sn.replace('/','.').split('[')[0]: col for col in dfemb_1.columns for sn in col.split(',')}
dfemb_2.columns = [sn2fullsn[sn] for sn in dfemb_2.columns]
dfemb_2


[Warning] Repeated npi_init (called 7 times). Non first-time-use npi_init will be ignored until using npi_end.


,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_hellaCacheArb_io_requestor_0_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/ptw/io_mem_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_requestor_0_s2_nack","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_lsu_io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_hellacache_s2_nack,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/hellaCacheArb/io_mem_s2_nack",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid_0,"TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_valid,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_valid","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_ldq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_ldq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_stq_idx[4:0],TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_stq_idx[4:0]","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_ldq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_ldq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_uop_uses_stq,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_uop_uses_stq","TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/_dcache_io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/io_lsu_nack_0_bits_is_hella,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/io_dmem_nack_0_bits_is_hella",TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/dcache/s1_send_resp_or_nack_0,...,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_14,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_15,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_16,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_17,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_18,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_19,TestDriver/testHarness/chiptop0/system/tile_prci_domain/element_reset_domain_boom_tile/lsu/older_nacked_REG_20,TestDriver/testHarness/chiptop0/s

In [25]:
# # actual_sigval = df
# c = actual_sigval.columns[0]
# c

In [26]:
df1 = dfemb_1.copy()
df1.index = [int(i.split('.')[1]) for i in dfemb_1.index]
idx = [i for i in dfemb_2.index if i != 0 and i in df1.index]
df1 = df1.loc[idx]
df2 = dfemb_2.loc[idx]

In [27]:
# df1[mismatched_cols[0]].values,
# df2[mismatched_cols[0]].values
# # , fsdbdata.siginfo

In [28]:
diff = (df1.astype(np.int32) - df2.astype(np.int32))
mismatched_cols = diff.columns[(diff != 0).any()]
mismatched_rows = diff.index[(diff != 0).any(axis=1)]
# diff[mismatched_cols]
print(len(mismatched_cols))
if len(mismatched_cols) > 0:
    display(diff.loc[mismatched_rows,mismatched_cols],
        df1.loc[mismatched_rows,mismatched_cols],
        df2.loc[mismatched_rows,mismatched_cols],
        fsdbdata.siginfo.reset_index().set_index('signame').loc[mismatched_cols])

0
